In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn

import os
import warnings

from IPython.display import display

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
class SelfAttention(nn.Module):
    """
    Single-head self-attention module, used to train the following parameters:
    - Q matrix: query vectors;
    - K matrix: key vectors;
    - V matrix: value vectors.
    """

    def __init__(self, dim: int = 256) -> None:
        super().__init__()

        self.scale = dim ** 0.5

        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        weights = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / self.scale, dim=-1)

        return torch.matmul(weights, V)

In [4]:
class WordEncoder(nn.Module):
    """
    
    """
    def __init__(self, vocab_size, embedding_dim=32, hidden=128, pad_id=0) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embedding_dim, hidden, batch_first=True, bidirectional=True)


    def forward(self, x: torch.Tensor) -> torch.Tensor:

        batch, words, chars = x.shape           # [batch, words, chars]

        x = x.view(batch * words, chars)        # [batch * words, chars]
        emb = self.embedding(x)                 # [batch * words, chars, emb_size]


        
        _, (h, _) = self.lstm(emb)              # [2, batch * words, hidden_size]
        h = torch.cat([h[0], h[1]], dim=-1)     # [batch * words, 2 * hidden_size]
        h = h.view(batch, words, -1)            # [batch, words, 2 * hidden_size]

        return h

In [5]:
class SentenceEncoder(nn.Module):
    """
    
    """

    def __init__(self, input_dim=256, hidden=128) -> None:
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden, batch_first=True, bidirectional=True)
        self.attention = SelfAttention()


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        out, _ = self.lstm(x)                   # [batch, words, 2 * hidden_size]
        out = self.attention(out)
        return out

In [6]:
x = torch.tensor([
    [
        [1, 2, 3, 0, 0],   # word 1     |
        [4, 5, 6, 7, 0],   # word 2     } sentence 1
        [8, 9, 10, 11, 12] # word 3     |
    ],
    [
        [3, 4, 0, 0, 0],   #            |
        [7, 8, 9, 0, 0],   #            } sentence 2
        [2, 3, 4, 5, 6]    #            |
    ],
])

In [7]:
class POSTagger(nn.Module):
    """
    
    """

    def __init__(
            self,
            vocab_size,
            num_tags,
            embedding_dim=128,
            word_hidden=128,
            sent_hidden=128,
            pad_id=0,
    ) -> None:
        super().__init__()

        self.word_encoder = WordEncoder(
            vocab_size, embedding_dim, word_hidden, pad_id
        ) # [word1vec, word2vec, word3vec]

        self.sent_encoder = SentenceEncoder(
            2 * word_hidden, sent_hidden
        ) # [word1|conеxt, word2|context, word3|context]
        self.linear = nn.Linear(
            2 * sent_hidden, num_tags
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:

        word_repr = self.word_encoder(x)
        word_repr_context = self.sent_encoder(word_repr)
        logits = self.linear(word_repr_context)

        return logits

In [8]:
from torch.utils.data import Dataset, DataLoader

In [9]:
def read_conllu(path):
    """
    
    """
    sentences = []
    lemmas_all = []
    pos_all = []

    with open(path, encoding="utf-8") as f:
        words, lemmas, pos_tags = [], [], []

        for line in f:
            line = line.strip()

            if not line:
                if words:
                    sentences.append(words)
                    lemmas_all.append(lemmas)
                    pos_all.append(pos_tags)
                    words, lemmas, pos_tags = [], [], []
                continue

            if line.startswith("#"):
                continue

            parts = line.split("\t")

            if "-" in parts[0] or "." in parts[0]:
                continue

            words.append(parts[1])
            lemmas.append(parts[2])
            pos_tags.append(parts[3])

    return sentences, lemmas_all, pos_all

In [10]:
train = read_conllu('../data/train.conllu')
test = read_conllu('../data/test.conllu')

In [11]:
words  = train[0]
lemmas = train[1]
tags   = train[2]

display(words[0][2])
display(lemmas[0][2])
display(tags[0][2])

'хоча'

'хацець'

'VERB'

In [12]:
def build_char_vocab(sentences):
    """
    char  -> idx
    <PAD> -> 0
    """
    chars = set()
    for sentence in sentences:
        for word in sentence:
            chars.update(list(word))

    vocabulary = {char: i + 2 for i, char in enumerate(chars)}
    vocabulary["<PAD>"] = 0
    vocabulary["<UNK>"] = 1
    return vocabulary

def build_pos_vocab(sentences):
    """
    tag   -> idx
    <PAD> -> -100
    """
    tags = set(tag for sentence in sentences for tag in sentence)
    vocabulary = {tag: i for i, tag in enumerate(tags)}
    vocabulary["<PAD>"] = -100
    return vocabulary

In [13]:
class POSDataset(Dataset):
    def __init__(self, sentences, pos_tags, char2idx, tag2idx):
        self.sentences = sentences
        self.pos_tags = pos_tags
        self.char2idx = char2idx
        self.tag2idx = tag2idx

    def __len__(self):
        return len(self.sentences)
    
    def encode_word(self, word: str) -> list[int]:
        return [self.char2idx.get(ch, self.char2idx["<UNK>"]) for ch in word]

    def __getitem__(self, index):
        words = self.sentences[index]
        tags = self.pos_tags[index]
        
        sentences_encoded = [self.encode_word(word) for word in words]
        tags_encoded = [self.tag2idx[tag] for tag in tags]

        return sentences_encoded, tags_encoded

In [14]:
conditions = torch.load('../models/POSTagger.pt', map_location='cpu', weights_only=False)

In [38]:
conditions.keys()

dict_keys(['tagger_state_dict', 'char2idx', 'tag2idx'])

In [15]:
# char2idx = build_char_vocab(train[0])
# tag2idx = build_pos_vocab(train[2])

char2idx = conditions['char2idx']
tag2idx = conditions['tag2idx']

train_dataset = POSDataset(
    train[0], train[2], char2idx, tag2idx
)

test_dataset = POSDataset(
    test[0], test[2], char2idx, tag2idx
)

In [16]:
def collate_fn(batch) -> tuple[torch.Tensor, torch.Tensor]:
    sentences, tags = zip(*batch)

    max_word_len = max(len(w) for s in sentences for w in s)
    max_sent_len = max(len(s) for s in sentences)

    padded_sentences = []
    padded_tags = []

    for sent, tag_seq in zip(sentences, tags):

        padded_sent = []
        for word in sent:
            padded_word = word + [0] * (max_word_len - len(word))
            padded_sent.append(padded_word)

        for _ in range(max_sent_len - len(sent)):
            padded_sent.append([0] * max_word_len)

        padded_sentences.append(padded_sent)

    
        padded_tag_seq = tag_seq + [-100] * (max_sent_len - len(tag_seq))
        padded_tags.append(padded_tag_seq)

    return (
        torch.tensor(padded_sentences, dtype=torch.long),
        torch.tensor(padded_tags, dtype=torch.long)
    )

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    collate_fn=collate_fn
)


In [19]:
def train_step(model, batch, optimizer, criterion, device):
    model.train()

    x, y = batch[0].to(device), batch[1].to(device)
    
    optimizer.zero_grad()

    logits = model(x)

    loss = criterion(
        logits.view(-1, logits.size(-1)),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    return loss.item()


def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            logits = model(x)

            loss = criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)


def accuracy(logits, y):
    preds = logits.argmax(-1)
    mask = y != -100
    return ((preds == y) & mask).sum() / mask.sum()

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = POSTagger(len(char2idx), len(tag2idx) - 1)

epochs = 20
batch_size = 16
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.to(device)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn
)

In [ ]:
avg_train_loss = float("inf")
val_loss = float("inf")
val_acc = 0

for epoch in range(epochs):
    # ---- TRAIN ----
    total_train_loss = 0

    for i, batch in enumerate(train_loader):
        if i % 100 == 0:
            clear_output(wait=True)
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"Train loss: {avg_train_loss:.4f}")
            print(f"Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")
            print("-" * 36)
            print(f"{i} / {len(train_loader)}")
        loss = train_step(model, batch, optimizer, criterion, device)
        total_train_loss += loss

    avg_train_loss = total_train_loss / len(train_loader)

    # ---- VALIDATION ----
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

START
0 / 1429


KeyboardInterrupt: 

In [18]:
# WARNING 

model = POSTagger(len(char2idx), len(tag2idx) - 1)
model.load_state_dict(conditions['tagger_state_dict'])

<All keys matched successfully>

In [19]:
def predict(model, sentence: list[str], char2idx, device):
    model.eval()
    max_word_length = max(len(w) for w in sentence)
    padded = []

    for word in sentence:
        encoded = [char2idx.get(ch, char2idx["<UNK>"]) for ch in word]
        encoded += [char2idx["<PAD>"]] * (max_word_length - len(word))
        padded.append(encoded)

    x = torch.tensor([padded], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits[0], dim=-1).max(-1)
        preds = logits.argmax(-1)[0]

    return probs

In [20]:
idx2tag = {v: k for k, v in tag2idx.items()}

In [25]:
text_split = """
Фасады гарызантальна ашаляваны , прарэзаны лучковымі аконнымі праёмамі .
""".split()
probs, tags = predict(model, text_split, char2idx, device)

print("word           tag      confidence")
for word, prob, tag in zip(text_split, probs, tags):
    print(f"{word:<14} {idx2tag[tag.item()]:<8}   {prob.item():.3f}")

word           tag      confidence
Фасады         NOUN       0.703
гарызантальна  ADV        0.565
ашаляваны      VERB       0.990
,              PUNCT      0.836
прарэзаны      VERB       0.917
лучковымі      ADJ        0.990
аконнымі       ADJ        0.996
праёмамі       NOUN       0.996
.              PUNCT      0.769


Проблемы:

сейчас модель просто ужасно работает с короткими предложениями. 
- усложнить/упростить модель немного большим числом скрытых слоев
- data augmentation (очень очень нужно): срезы, сэмплинг, lowercase, убирать пунктуацию и тп
- подумать, стоит ли пытаться устранить необходимость в разметке знаков препинания (полностью)
